# FileIOUtils tutorial

`FileIOUtils` の基本的な使い方を確認するための Notebook。

この Notebook は一時ディレクトリにサンプルファイルを作成し、CSV / parquet / YAML / pickle の読み書き、`FileConfigRegistry`、placeholder 解決、wildcard 読み込み、dtype 変換を順に実行する。

In [ ]:
from __future__ import annotations

import logging
import tempfile
from pathlib import Path
from typing import Any

import pandas as pd

from myproj.io.config_resolver import (
    find_project_root,
    load_dataset_definition,
    resolve_placeholders,
)
from myproj.io.file_io import (
    CsvReadOptions,
    FileConfigRegistry,
    FileIOUtils,
    FileReadConfig,
    FileSpec,
    FileWriteConfig,
    ReadOptions,
    WriteOptions,
)

In [ ]:
DATASET_YAML = Path("shared/py/myproj/conf/dataset/completejourney/10_interim.yaml")
OUTPUT_CONFIG_YAML = Path("articles/_template/conf/01_file_io_utils_tutorial/99_output.yaml")
LOGGER_NAME = "file_io_utils_tutorial"

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")
project_root = find_project_root(Path.cwd())
temp_dir = tempfile.TemporaryDirectory()
work_dir = Path(temp_dir.name)

logger = logging.getLogger(LOGGER_NAME)
file_io = FileIOUtils(logger)

print(f"project_root: {project_root}")
print(f"work_dir: {work_dir}")

In [ ]:
def build_sample_frame() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "customer_id": [1, 2, 3],
            "ordered_at": ["2026-01-01", "2026-01-02", "2026-01-03"],
            "sales_value": [1200.5, 900.0, 1500.25],
            "segment": ["A", "B", "A"],
        }
    )


def print_frame_summary(name: str, frame: pd.DataFrame) -> None:
    print(f"{name}: shape={frame.shape}")
    display(frame)

## 1. dataclass config を直接組み立てて読み書きする

In [ ]:
sample = build_sample_frame()

csv_write_config = FileWriteConfig(
    file=FileSpec(path=work_dir, name="orders.csv", type="csv"),
    options=WriteOptions(),
)
csv_read_config = FileReadConfig(
    file=csv_write_config.file,
    options=ReadOptions(),
)

file_io.write_file(sample, csv_write_config)
loaded_csv = file_io.read_file(csv_read_config)
print_frame_summary("orders.csv", loaded_csv)

In [ ]:
parquet_write_config = FileWriteConfig(
    file=FileSpec(path=work_dir, name="orders.parquet", type="parquet"),
    options=WriteOptions(),
)
parquet_read_config = FileReadConfig(
    file=parquet_write_config.file,
    options=ReadOptions(),
)

file_io.write_file(sample, parquet_write_config)
loaded_parquet = file_io.read_file(parquet_read_config)
print_frame_summary("orders.parquet", loaded_parquet)

## 2. YAML / pickle を読み書きする

In [ ]:
metadata: dict[str, Any] = {
    "dataset": "orders",
    "columns": ["customer_id", "ordered_at", "sales_value", "segment"],
}

yaml_config = FileWriteConfig(
    file=FileSpec(path=work_dir, name="metadata.yaml", type="yaml"),
    options=WriteOptions(),
)
file_io.write_file(metadata, yaml_config)
loaded_yaml = file_io.read_file(
    FileReadConfig(file=yaml_config.file, options=ReadOptions())
)
print(loaded_yaml)

In [ ]:
pickle_config = FileWriteConfig(
    file=FileSpec(path=work_dir, name="metadata.pkl", type="pickle"),
    options=WriteOptions(),
)
file_io.write_file(metadata, pickle_config)
loaded_pickle = file_io.read_file(
    FileReadConfig(file=pickle_config.file, options=ReadOptions())
)
print(loaded_pickle)

## 3. FileConfigRegistry で default + 個別設定をマージして読む

In [ ]:
sample.to_parquet(work_dir / "orders_from_registry.parquet")

config_definition = {
    "default": {
        "file": {
            "path": str(work_dir),
            "name": "",
            "type": "parquet",
        }
    },
    "orders": {
        "file": {
            "name": "orders_from_registry.parquet",
        }
    },
}

registry = FileConfigRegistry.from_mapping(config_definition)
loaded_from_registry = file_io.read_file(registry.read_config("orders"))
print_frame_summary("orders_from_registry.parquet", loaded_from_registry)

## 4. FileConfigRegistry.write_config() で出力設定を管理する

出力用設定は `articles/_template/conf/01_file_io_utils_tutorial/99_output.yaml` に置く。`{tutorial_output_dir}` は実行時の出力先ディレクトリに置換する。

In [ ]:
metadata = {
    "dataset": "orders",
    "rows": len(sample),
    "columns": sample.columns.tolist(),
}
output_definition = load_dataset_definition(
    project_root / OUTPUT_CONFIG_YAML,
    project_root,
    extra_placeholders={"tutorial_output_dir": str(work_dir)},
)
output_registry = FileConfigRegistry.from_mapping(output_definition)

file_io.write_file(sample, output_registry.write_config("orders_tsv"))
file_io.write_file(sample, output_registry.write_config("orders_parquet"))
file_io.write_file(metadata, output_registry.write_config("orders_metadata"))

output_dir = output_registry.write_config("orders_tsv").file.path
print(sorted(path.name for path in output_dir.iterdir()))

In [ ]:
tsv_read_config = FileReadConfig(
    file=output_registry.write_config("orders_tsv").file,
    options=ReadOptions(csv=CsvReadOptions(delimiter="\t")),
)
loaded_tsv = file_io.read_file(tsv_read_config)
loaded_metadata = file_io.read_file(
    FileReadConfig(
        file=output_registry.write_config("orders_metadata").file,
        options=ReadOptions(),
    )
)

print_frame_summary("orders.tsv", loaded_tsv)
print(loaded_metadata)

## 5. config_resolver で placeholder を解決する

In [ ]:
resolved = resolve_placeholders(
    {
        "path": "{path_sys_base}/shared/data",
        "unknown": "{not_defined}/path",
    },
    {"path_sys_base": str(project_root)},
)

print(f"resolved path: {resolved['path']}")
print(f"unknown placeholder remains: {resolved['unknown']}")

if (project_root / DATASET_YAML).exists():
    completejourney_definition = load_dataset_definition(
        project_root / DATASET_YAML,
        project_root,
    )
    print(
        "completejourney default path: "
        f"{completejourney_definition['default']['file']['path']}"
    )

## 6. wildcard を展開して複数ファイルをまとめて読む

In [ ]:
for index in range(1, 4):
    frame = pd.DataFrame(
        {
            "batch": [index, index],
            "value": [index * 10, index * 10 + 1],
        }
    )
    frame.to_csv(work_dir / f"batch_{index}.csv", index=False)

wildcard_config = FileReadConfig(
    file=FileSpec(path=work_dir, name="batch_*.csv", type="csv"),
    options=ReadOptions(),
)
expanded_config = file_io.expand_wildcards(wildcard_config)
loaded_batches = file_io.read_files(expanded_config, concat=True)

print(f"expanded names: {expanded_config.file.names}")
print_frame_summary("batch_*.csv", loaded_batches)

## 7. convert_dtype で読み込み後の型を整える

In [ ]:
typed_frame = build_sample_frame()
file_io.convert_dtype(
    typed_frame,
    {
        "datetime64": {"ordered_at": "%Y-%m-%d"},
        "int64": ["customer_id"],
        "float64": ["sales_value"],
    },
)

typed_frame.dtypes

## 後片付け

一時ディレクトリを使った場合は、最後に cleanup する。固定ディレクトリで試したい場合は、この cleanup は不要。

In [ ]:
temp_dir.cleanup()